In [ ]:
### Identify uncertain/confident viruses that are not LQ from release r2025_09 and r2025_10
import polars as pl

mine_report = pl.read_csv('viruses.csvtk_concat.tsv', separator='\t', null_values='NA')

non_lq_viruses = (
    mine_report
        .filter(
            (pl.col('kmer_freq') <= 1.2) & 
            (
                ((~pl.col('warningsPrediction').str.contains('>1 viral region detected')) & (~pl.col('warningsPrediction').str.contains('contig >1.5x longer'))) |
                (pl.col('warningsPrediction').is_null())
            ) &
            ((pl.col('completeness') >= 50) | (pl.col('contig_length') >= 10000)) &
            (pl.col('taxonomy') != 'Unclassified') &
            ((pl.col('taxonomy').str.contains('viricetes')) | (pl.col('taxonomy').str.contains('Anelloviridae')))
        )
        .with_columns([
            pl.when(pl.col('source_db').is_not_null()).then(pl.col('source_db'))
                .when(pl.col('seq_name').str.contains('\.contig')).then(pl.lit('ATB'))
                .when(pl.col('seq_name').str.contains('@')).then(pl.lit('CHVD'))
                .when(pl.col('seq_name').str.contains(r'^v\d')).then(pl.lit('CNGVC'))
                .when(pl.col('seq_name').str.contains('_vae_')).then(pl.lit('CNGVR'))
                .when(pl.col('seq_name').str.contains(r'^ERZ')).then(pl.lit('ENA'))
                .when(pl.col('seq_name').str.contains(r'^IMGVR')).then(pl.lit('IMGVR'))
                .when(pl.col('seq_name').str.contains(r'_round')).then(pl.lit('MMGE'))
                .when(pl.col('seq_name').str.contains(r'^(S|E|D)RR\d+_\d+$')).then(pl.lit('LOGAN'))
                .when(pl.col('seq_name').str.contains(r'^(S|E|D)RR\d+_\d+\|provirus')).then(pl.lit('LOGAN'))
                .when(pl.col('seq_name').str.contains(r'^opdg_')).then(pl.lit('OPD'))
                .when(pl.col('seq_name').str.contains(r'_cf_k')).then(pl.lit('PRJ'))
                .when(pl.col('seq_name').str.contains(r'^SMGC_')).then(pl.lit('SMGC'))
                .when(pl.col('seq_name').str.contains(r'_k\d+_\d+$')).then(pl.lit('SPIRE'))
                .when(pl.col('seq_name').str.contains(r'_k\d+_\d+\|provirus')).then(pl.lit('SPIRE'))
                .when(pl.col('seq_name').str.contains(r'^[a-zA-Z0-9]+\.k1.1_\d+')).then(pl.lit('VMGC'))
                .when(pl.col('seq_name').str.contains(r'_\d+\.k\d+_\d+')).then(pl.lit('OVD'))
                .alias('source_db')
        ])
        .filter(pl.col('uhvdb_virus_classification') != 'non-viral')
)

(
    non_lq_viruses
        .filter(pl.col('source_db') != 'ATB')
        .filter(~pl.col('source_db').str.contains(r'^PRJ'))
        [['seq_name']]
        .write_csv('r2025_09_non_lq_viruses.csv', include_header=False)
)

(
    non_lq_viruses
        .filter((pl.col('source_db') != 'ATB') | (pl.col('source_db').str.contains(r'^PRJ')))
        [['seq_name']]
        .write_csv('r2025_10_non_lq_viruses.csv', include_header=False)
)
        

In [ ]:
!seqkit \
    grep \
    /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/uhvdb_r2025_09_all_uhvdb_virus.fna \
    --pattern-file /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/r2025_09_non_lq_viruses.csv \
    -j 4 \
    --out-file /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/r2025_09_non_lq_viruses.fna.gz

In [ ]:
!nextflow run /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit \
    -profile uw_hyak \
    -w /gscratch/scrubbed/carsonjm/2026.03.16 \
    --input uhgv_hq_hc_samplesheet.csv \
    --db_dir /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases \
    --checkv_db /gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit/databases/checkv/uhgv_1 \
    --output_dir uhgv_hq_hc_results \
    --hyak_partition="stf" \
    --hyak_queue="ckpt" \
    --new_release_id="2026-03-16" \
    --run_classify=true \
    --run_hqfilter=true \
    --run_hcfilter=true